In [0]:
# Celda 1
%pip install statsmodels

In [0]:
# Celda 2
dbutils.library.restartPython()

In [0]:
# Celda 3: datos y listas de variables
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import roc_auc_score, roc_curve
import mlflow

df = spark.table("workspace.default.lc_woe").toPandas()
lista = spark.table("workspace.default.lc_model_variables").toPandas()

vars_a = [f"woe_{v}" for v in lista["variable"]]
vars_b = [f"woe_{v}" for v in lista.loc[lista["modelo_b_06"], "variable"]]

dev = df[df["sample"] == "DEV"]
oot = df[df["sample"] == "OOT"]
print(f"Modelo A: {len(vars_a)} vars | Modelo B: {len(vars_b)} vars | DEV: {len(dev):,} | OOT: {len(oot):,}")


In [0]:
# Celda 4: funciones de evaluación
def ks_auc(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return round(max(tpr - fpr) * 100, 2), round(roc_auc_score(y, p), 4)   # KS en %, AUC

def vif(X):
    Xc = sm.add_constant(X)
    return pd.Series([variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
                     index=X.columns).round(2)

def confusion(y, p, tasa_rechazo):
    corte = np.quantile(p, 1 - tasa_rechazo)   # p = probabilidad de malo: se rechaza el % más alto
    rech = p >= corte
    tp = int((rech & (y == 1)).sum());  fp = int((rech & (y == 0)).sum())
    fn = int((~rech & (y == 1)).sum()); tn = int((~rech & (y == 0)).sum())
    return {"tasa_rechazo": tasa_rechazo,
            "tasa_real_pct": round(rech.mean() * 100, 2),
            "malos_rechazados": tp, "buenos_rechazados": fp,
            "malos_aprobados": fn, "buenos_aprobados": tn,
            "recall_malos_pct": round(tp / (tp + fn) * 100, 2),
            "precision_rechazo_pct": round(tp / (tp + fp) * 100, 2),
            "buenos_rechazados_pct": round(fp / (fp + tn) * 100, 2),
            "bad_rate_aprobados_pct": round(fn / (fn + tn) * 100, 2)}


In [0]:
# Celda 5: entrenar, evaluar y registrar en MLflow
def entrenar(nombre, variables):
    with mlflow.start_run(run_name=nombre):
        y_dev, y_oot = dev["target"].values, oot["target"].values
        modelo = sm.Logit(y_dev, sm.add_constant(dev[variables])).fit(disp=0)

        p_dev = np.asarray(modelo.predict(sm.add_constant(dev[variables])))
        p_oot = np.asarray(modelo.predict(sm.add_constant(oot[variables])))
        ks_d, auc_d = ks_auc(y_dev, p_dev)
        ks_o, auc_o = ks_auc(y_oot, p_oot)

        coefs = pd.DataFrame({"coef": modelo.params.round(4), "p_value": modelo.pvalues.round(4)})
        coefs["vif"] = vif(dev[variables])
        conf = pd.DataFrame([confusion(y_oot, p_oot, t) for t in (0.10, 0.20, 0.30)])

        mlflow.log_params({"n_variables": len(variables), "variables": ",".join(variables)})
        mlflow.log_metrics({"ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o})
        mlflow.log_table(coefs.reset_index(names="variable"), "coeficientes.json")
        mlflow.log_table(conf, "confusion_oot.json")

    resumen = {"modelo": nombre, "n_vars": len(variables),
               "ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o}
    return modelo, coefs, conf, resumen

mod_a, coef_a, conf_a, res_a = entrenar("Modelo A - r0.7 - 16 vars", vars_a)
mod_b, coef_b, conf_b, res_b = entrenar("Modelo B - r0.6 - 14 vars", vars_b)

display(pd.DataFrame([res_a, res_b]))


In [0]:
print(pd.DataFrame([res_a, res_b]).to_string())
print(coef_a.to_string())
print(coef_b.to_string())
print(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B")]).to_string())


In [0]:
# Celda 6: coeficientes con p-value y VIF, y matrices de confusión en OOT
display(coef_a.reset_index(names="variable").assign(modelo="A"))
display(coef_b.reset_index(names="variable").assign(modelo="B"))
display(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B")]))


In [0]:
# Celda 8: modelo C = A sin mort_acc (no significativa) ni revol_bal (signo invertido)
vars_c = [v for v in vars_a if v not in ("woe_mort_acc", "woe_revol_bal")]
mod_c, coef_c, conf_c, res_c = entrenar("Modelo C - A sin mort_acc ni revol_bal - 14 vars", vars_c)

print(pd.DataFrame([res_a, res_b, res_c]).to_string())
print(coef_c.to_string())
print(conf_c.to_string())

#display(pd.concat([conf_a.assign(modelo="A"), conf_b.assign(modelo="B"), conf_b.assign(modelo="C")]))

In [0]:
# Celda 9: árbol con restricciones y elección de profundidad con validación cruzada en DEV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold

X_dev, y_dev = dev[vars_c].to_numpy(), dev["target"].to_numpy()
X_oot, y_oot = oot[vars_c].to_numpy(), oot["target"].to_numpy()
MIN_HOJA_PCT = 0.05   # cada hoja con al menos el 5% de la muestra de entrenamiento

def construir(X, y, idx, usadas, prof, max_prof, min_hoja):
    nodo = {"n": len(idx), "bad_rate": float(y[idx].mean())}
    libres = [j for j in range(X.shape[1]) if j not in usadas]   # condición 1: no repetir en la rama
    if prof >= max_prof or len(idx) < 2 * min_hoja or not libres:
        return nodo
    tocon = DecisionTreeClassifier(max_depth=1, min_samples_leaf=min_hoja, random_state=42)
    tocon.fit(X[np.ix_(idx, libres)], y[idx])
    if tocon.tree_.node_count == 1:   # ningún corte cumple el tamaño mínimo de hoja
        return nodo
    j = libres[tocon.tree_.feature[0]]
    corte = float(tocon.tree_.threshold[0])
    m = X[idx, j] <= corte
    nodo.update({"var": j, "corte": corte,
                 "izq": construir(X, y, idx[m], usadas | {j}, prof + 1, max_prof, min_hoja),
                 "der": construir(X, y, idx[~m], usadas | {j}, prof + 1, max_prof, min_hoja)})
    return nodo

def ajustar(X, y, max_prof):
    return construir(X, y, np.arange(len(y)), frozenset(), 0, max_prof, int(MIN_HOJA_PCT * len(y)))

def predecir(arbol, X):
    p, hoja, k = np.empty(len(X)), np.empty(len(X), dtype=int), [0]
    def rec(n, idx):
        if "var" not in n:
            p[idx], hoja[idx] = n["bad_rate"], k[0]   # probabilidad = bad rate de la hoja en DEV
            k[0] += 1
            return
        m = X[idx, n["var"]] <= n["corte"]
        rec(n["izq"], idx[m]); rec(n["der"], idx[~m])
    rec(arbol, np.arange(len(X)))
    return p, hoja

# Validación cruzada de 5 particiones para elegir la profundidad
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas = []
for d in range(2, 8):
    aucs = [roc_auc_score(y_dev[va], predecir(ajustar(X_dev[tr], y_dev[tr], d), X_dev[va])[0])
            for tr, va in skf.split(X_dev, y_dev)]
    filas.append((d, round(np.mean(aucs), 4)))

cv = pd.DataFrame(filas, columns=["max_depth", "auc_cv_dev"])
print(cv.to_string())
mejor_d = int(cv.loc[cv["auc_cv_dev"].idxmax(), "max_depth"])   # ante empate, gana el menos profundo
print("Mejor profundidad:", mejor_d)



In [0]:
# Celda 10: árbol final, reglas, estabilidad por hoja y comparación con el campeón
arbol = ajustar(X_dev, y_dev, mejor_d)
p_dev, h_dev = predecir(arbol, X_dev)
p_oot, h_oot = predecir(arbol, X_oot)

# Reglas del árbol, numeradas en el mismo orden que las hojas
k = [0]
def reglas(n, nivel=0):
    pref = "    " * nivel
    if "var" not in n:
        print(f"{pref}HOJA {k[0]}: n={n['n']:,}  bad_rate={n['bad_rate'] * 100:.2f}%"); k[0] += 1
        return
    v = vars_c[n["var"]]
    print(f"{pref}{v} <= {n['corte']:.4f}"); reglas(n["izq"], nivel + 1)
    print(f"{pref}{v} >  {n['corte']:.4f}"); reglas(n["der"], nivel + 1)
reglas(arbol)

# Distribución, bad rate y PSI por hoja (DEV contra OOT)
n_hojas = h_dev.max() + 1
hojas = pd.DataFrame({"hoja": range(n_hojas),
                      "pct_dev": np.bincount(h_dev, minlength=n_hojas) / len(h_dev),
                      "pct_oot": np.bincount(h_oot, minlength=n_hojas) / len(h_oot)})
hojas["bad_dev_pct"] = [y_dev[h_dev == i].mean() * 100 for i in hojas["hoja"]]
hojas["bad_oot_pct"] = [y_oot[h_oot == i].mean() * 100 for i in hojas["hoja"]]
hojas["psi"] = (hojas["pct_oot"] - hojas["pct_dev"]) * np.log(hojas["pct_oot"] / hojas["pct_dev"])
print(hojas.round(4).to_string())
print(f"PSI total: {hojas['psi'].sum():.4f}")

# Métricas y MLflow
ks_d, auc_d = ks_auc(y_dev, p_dev)
ks_o, auc_o = ks_auc(y_oot, p_oot)
conf_t = pd.DataFrame([confusion(y_oot, p_oot, t) for t in (0.10, 0.20, 0.30)])

with mlflow.start_run(run_name=f"Constrained Tree - depth {mejor_d} - leaf 5%"):
    mlflow.log_params({"max_depth": mejor_d, "min_leaf_pct": MIN_HOJA_PCT, "n_hojas": int(n_hojas),
                       "regla": "variable no se repite dentro de una rama"})
    mlflow.log_metrics({"ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o,
                        "psi_hojas": float(hojas["psi"].sum())})
    mlflow.log_table(conf_t, "confusion_oot.json")

res_t = {"modelo": f"Constrained Tree depth {mejor_d}", "n_vars": len(vars_c),
         "ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o}
print(pd.DataFrame([res_c, res_t]).to_string())
print(conf_t.to_string())


In [0]:
# Celda 11: AUC parcial en las zonas de operación (FPR = % de buenos rechazados)
from sklearn.metrics import auc as area

def pauc(y, p, max_fpr):
    fpr, tpr, _ = roc_curve(y, p)
    corte = np.searchsorted(fpr, max_fpr, side="right")
    x = np.append(fpr[:corte], max_fpr)                       # curva ROC hasta el límite de FPR
    t = np.append(tpr[:corte], np.interp(max_fpr, fpr, tpr))
    crudo = area(x, t)
    estandarizado = roc_auc_score(y, p, max_fpr=max_fpr)       # McClish: 0,5 = azar, 1 = perfecto
    return round(crudo, 4), round(estandarizado, 4)

# Probabilidades del modelo C (campeón) y del árbol
pc_dev = np.asarray(mod_c.predict(sm.add_constant(dev[vars_c])))
pc_oot = np.asarray(mod_c.predict(sm.add_constant(oot[vars_c])))

filas = []
for nombre, y, p, muestra in [("C logística", y_dev, pc_dev, "DEV"), ("C logística", y_oot, pc_oot, "OOT"),
                              ("Árbol", y_dev, p_dev, "DEV"), ("Árbol", y_oot, p_oot, "OOT")]:
    for lim in (0.10, 0.20, 0.30):
        crudo, est = pauc(y, p, lim)
        filas.append((nombre, muestra, lim, crudo, est))

pauc_tabla = pd.DataFrame(filas, columns=["modelo", "muestra", "max_fpr", "pauc_crudo", "pauc_estandarizado"])
print(pauc_tabla.to_string())


In [0]:
# Celda 12: modelo D, logística con las 5 variables de mayor IV
vars_d = vars_c[:5]
print("Variables D:", vars_d)

mod_d, coef_d, conf_d, res_d = entrenar("Modelo D - top 5 IV - 5 vars", vars_d)
pd_dev = np.asarray(mod_d.predict(sm.add_constant(dev[vars_d])))
pd_oot = np.asarray(mod_d.predict(sm.add_constant(oot[vars_d])))
print(coef_d.to_string())


In [0]:
# Celda 13: modelo E, árbol con restricciones sobre la probabilidad de D + 9 variables restantes
resto = [v for v in vars_c if v not in vars_d]
nombres_e = ["prob_modelo_D"] + resto
Xe_dev = np.column_stack([pd_dev, dev[resto].to_numpy()])
Xe_oot = np.column_stack([pd_oot, oot[resto].to_numpy()])

# Profundidad con validación cruzada en DEV (mismas reglas: hoja >= 5%, sin repetir variable en la rama)
filas = []
for d in range(2, 8):
    aucs = [roc_auc_score(y_dev[va], predecir(ajustar(Xe_dev[tr], y_dev[tr], d), Xe_dev[va])[0])
            for tr, va in skf.split(Xe_dev, y_dev)]
    filas.append((d, round(np.mean(aucs), 4)))
cv_e = pd.DataFrame(filas, columns=["max_depth", "auc_cv_dev"])
print(cv_e.to_string())
d_e = int(cv_e.loc[cv_e["auc_cv_dev"].idxmax(), "max_depth"])

arbol_e = ajustar(Xe_dev, y_dev, d_e)
pe_dev, he_dev = predecir(arbol_e, Xe_dev)
pe_oot, he_oot = predecir(arbol_e, Xe_oot)

# Reglas
cont = [0]
def reglas_n(n, nombres, nivel=0):
    pref = "    " * nivel
    if "var" not in n:
        print(f"{pref}HOJA {cont[0]}: n={n['n']:,}  bad_rate={n['bad_rate'] * 100:.2f}%"); cont[0] += 1
        return
    v = nombres[n["var"]]
    print(f"{pref}{v} <= {n['corte']:.4f}"); reglas_n(n["izq"], nombres, nivel + 1)
    print(f"{pref}{v} >  {n['corte']:.4f}"); reglas_n(n["der"], nombres, nivel + 1)
reglas_n(arbol_e, nombres_e)

# PSI por hoja
nh = he_dev.max() + 1
pdv = np.bincount(he_dev, minlength=nh) / len(he_dev)
pot = np.bincount(he_oot, minlength=nh) / len(he_oot)
psi_e = float(((pot - pdv) * np.log(pot / pdv)).sum())

# Métricas, matriz de confusión y MLflow
ks_d, auc_d = ks_auc(y_dev, pe_dev)
ks_o, auc_o = ks_auc(y_oot, pe_oot)
conf_e = pd.DataFrame([confusion(y_oot, pe_oot, t) for t in (0.10, 0.20, 0.30)])
res_e = {"modelo": f"E híbrido D + árbol depth {d_e}", "n_vars": len(vars_c),
         "ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o}

with mlflow.start_run(run_name=f"Modelo E - hibrido D + arbol depth {d_e}"):
    mlflow.log_params({"max_depth": d_e, "min_leaf_pct": MIN_HOJA_PCT, "n_hojas": int(nh),
                       "entradas": ",".join(nombres_e)})
    mlflow.log_metrics({"ks_dev": ks_d, "auc_dev": auc_d, "ks_oot": ks_o, "auc_oot": auc_o, "psi_hojas": psi_e})
    mlflow.log_table(conf_e, "confusion_oot.json")

print(f"Hojas: {nh} | PSI hojas: {psi_e:.4f}")


In [0]:
# Celda 14: comparación final C vs D vs E en OOT
print(pd.DataFrame([res_c, res_d, res_e]).to_string())

filas = []
for nombre, p in [("C logística 14", pc_oot), ("D logística top 5", pd_oot), ("E híbrido", pe_oot)]:
    for lim in (0.10, 0.20, 0.30):
        filas.append((nombre, lim, pauc(y_oot, p, lim)[1]))
print(pd.DataFrame(filas, columns=["modelo", "max_fpr", "pauc_est"])
        .pivot(index="modelo", columns="max_fpr", values="pauc_est").to_string())

print(pd.concat([conf_c.assign(modelo="C"), conf_d.assign(modelo="D"), conf_e.assign(modelo="E")]).to_string())
